<h1 style="text-align: center;">Tektim Chatbot Model and Exploration</h3>

Raw data needed for preprocessing

1. Input: what prompted my message

2. Response: my message

3. Other features: other things that could influence my response

      - time of day

      - user

      - which server its in
    
    
Note: Due to dataset size it may be not in my best interest to make it too complex

In [1]:
#imports
import pandas as pd

In [48]:
df = pd.read_csv('data/break_room_testing/break_room_test.csv')
df_reversed = df.iloc[::-1].reset_index(drop=True)
df_reversed.to_csv('test_sheet_rev.csv', index=False)
df = df_reversed
df = df[['userName','content','date']]
df.head()

,userName,content,date
0,phioners,surely,2024-07-22T08:40:18.682Z
1,tigm,NaN,2024-07-22T08:50:19.430Z
2,tigm,holy fk,2024-07-22T08:50:20.783Z
3,tigm,one of my favorite pets,2024-07-22T08:50:25.951Z
4,xenltz,Wow nice,2024-07-22T09:34:45.206Z


In [49]:
rows_with_nan = df[df.isna().any(axis=1)]

rows_with_nan.head(5)
rows_with_nan.isna().sum()

userName     0
content     38
date         0
dtype: int64

In [52]:
fill_value = "[Image]"
df = df.fillna(value=fill_value)
df.head(20)

,userName,content,date
0,phioners,surely,2024-07-22T08:40:18.682Z
1,tigm,[Image],2024-07-22T08:50:19.430Z
2,tigm,holy fk,2024-07-22T08:50:20.783Z
3,tigm,one of my favorite pets,2024-07-22T08:50:25.951Z
4,xenltz,Wow nice,2024-07-22T09:34:45.206Z
5,xenltz,apparently ii had 3300 supremes in my bank tha...,2024-07-22T09:51:00.292Z
6,xenltz,?,2024-07-22T09:51:02.336Z
7,phioners,<:uhh:1037839657532145774>,2024-07-22T09:52:48.174Z
8,._.miles,Is dungeon good,2024-07-22T11:20:41.259Z
9,_musab,It drops moonstones,2024-07-22T11:20:58.823Z


In [56]:
# select content that does not have . ? !
# .*(?<![?!\.])$ <- regular expression to select it
selected_rows = df[df['content'].str.contains(r'.*(?<![?!\.])$', regex=True)]
selected_rows

,userName,content,date
0,phioners,surely,2024-07-22T08:40:18.682Z
1,tigm,[Image],2024-07-22T08:50:19.430Z
2,tigm,holy fk,2024-07-22T08:50:20.783Z
3,tigm,one of my favorite pets,2024-07-22T08:50:25.951Z
4,xenltz,Wow nice,2024-07-22T09:34:45.206Z
...,...,...,...
883,jibyjib,Yak was solo hm with mage,2024-07-29T04:53:24.173Z
884,jibyjib,Mammoth I don’t recall,2024-07-29T04:53:30.323Z
885,jibyjib,Yak having the ability to send shit back to th...,2024-07-29T04:53:58.669Z
886,jibyjib,But who cares about long trips anymore lmao,2024-07-29T04:54:11.378Z


This allowed me to discord NaN values, most likely images, in my dataset

In [ ]:
#example
df = df.drop(index=1)
df.reset_index(drop=True, inplace=True)
df.head()

## Attempt 1: ChatGPT-Neo

In [1]:
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

model = GPTNeoForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
tokenizer = GPT2Tokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")

c:\Users\timka\anaconda3\envs\tektim_model\lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
c:\Users\timka\anaconda3\envs\tektim_model\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [13]:
from datasets import load_dataset

# Set the pad_token to eos_token
tokenizer.pad_token = tokenizer.eos_token

def add_labels(example):
    example["labels"] = example["input_ids"].copy()  # Copy input_ids as labels
    return example

def tokenize_function(examples):
    return tokenizer(examples['input'], examples['output'], padding="max_length", truncation=True, max_length=512)

# Assume 'data' is your dataset of texts
# at this point, data needs to be created..
data = load_dataset('json', data_files='data\\text\\training_data.json')
tokenized_data = data['train'].map(tokenize_function, batched=True)

# Apply the function to add labels
tokenized_data = tokenized_data.map(add_labels)

tokenized_data = tokenized_data.train_test_split(test_size=0.2)  # 80:20 split
tokenized_data

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2
    })
    test: Dataset({
        features: ['input', 'output', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
})

In [14]:
# Setup Fine Tuning
from transformers import Trainer, TrainingArguments


# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    evaluation_strategy="epoch",     # evaluation frequency
    learning_rate=2e-5,              # learning rate
    per_device_train_batch_size=4,   # batch size for training
    per_device_eval_batch_size=4,    # batch size for evaluation
    num_train_epochs=3,              # number of training epochs
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
)

# Initialize Trainer with the model, arguments, and tokenized data
trainer = Trainer(
    model=model,                         # the model to train
    args=training_args,                  # training arguments
    train_dataset=tokenized_data['train'], 
    eval_dataset=tokenized_data['test'],
)

In [16]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.662796
2,No log,0.347524
3,No log,0.257549


TrainOutput(global_step=3, training_loss=0.2527049382527669, metrics={'train_runtime': 657.2062, 'train_samples_per_second': 0.009, 'train_steps_per_second': 0.005, 'total_flos': 22274245656576.0, 'train_loss': 0.2527049382527669, 'epoch': 3.0})

In [17]:
# Save the fine-tuned model
model.save_pretrained('data/models/raw/lang_model/fine_tuned_model')

# Save the tokenizer
tokenizer.save_pretrained('data/models/raw/lang_model/fine_tuned_model')

('data/models/raw/lang_model/fine_tuned_model\\tokenizer_config.json',
 'data/models/raw/lang_model/fine_tuned_model\\special_tokens_map.json',
 'data/models/raw/lang_model/fine_tuned_model\\vocab.json',
 'data/models/raw/lang_model/fine_tuned_model\\merges.txt',
 'data/models/raw/lang_model/fine_tuned_model\\added_tokens.json')

# Can Skip to here if the model is already created in raw/lang_model

In [26]:
# Load the fine-tuned model and tokenizer
model = GPTNeoForCausalLM.from_pretrained('data/models/raw/lang_model/fine_tuned_model')
tokenizer = GPT2Tokenizer.from_pretrained('data/models/raw/lang_model/fine_tuned_model')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [27]:
# Define the input for the model (new conversation)
input_text = "are you copying me"

# Tokenize the input
inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)

# Generate a response from the model
outputs = model.generate(
    inputs['input_ids'],
    attention_mask=inputs['attention_mask'],
    max_length=100,
    pad_token_id=tokenizer.pad_token_id)

# Decode the generated tokens into text
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# strip text of input
response = response[len(input_text):].strip()
print(response)

?

~~~
joshu
I'm not copying you, but I'm not sure what you mean by "I'm not copying you".

~~~
dang
I'm not sure what you mean either.
